In [22]:
# Written by Sebastian Matiz
import pandas as pd
import requests
import json
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error
from sklearn.metrics import accuracy_score

cols_to_drop_for_player_stats = [
    'comment',
    'player.firstname',
    'player.lastname',
    'player.id',
    'team.id',
    'team.nickname',
    'team.code',
    'team.name',
    'team.logo',
    'game.id',
    'pos'
]

cols_to_drop_for_game_stats = [
    'league',
    'season',
    'stage',
    'officials',
    'timesTied',
    'leadChanges',
    'nugget',
    'date.start',
    'date.end',
    'date.duration',
    'status.clock',
    'status.halftime',
    'status.short',
    'status.long',
    'periods.current',
    'periods.total',
    'periods.endOfPeriod',
    'arena.name',
    'arena.city',
    'arena.state',
    'arena.country',
    'teams.visitors.id',
    'teams.visitors.name',
    'teams.visitors.nickname',
    'teams.visitors.code',
    'teams.visitors.logo',
    'teams.home.id',
    'teams.home.name',
    'teams.home.nickname',
    'teams.home.code',
    'teams.home.logo',
    'scores.visitors.win',
    'scores.visitors.loss',
    'scores.visitors.series.win',
    'scores.visitors.series.loss',
    'scores.visitors.linescore',
    'scores.home.win',
    'scores.home.loss',
    'scores.home.series.win',
    'scores.home.series.loss',
    'scores.home.linescore'
]

In [77]:
# rapidApi headers
headers = {
    "X-RapidAPI-Key": "REDACTED_RAPIDAPI_KEY",
    "X-RapidAPI-Host": "api-nba-v1.p.rapidapi.com"
}

#########################################################
# team code by id
url = "https://api-nba-v1.p.rapidapi.com/teams"
response = requests.get(url, headers=headers).json()['response']
df = pd.DataFrame(response)
df = df.loc[(df['nbaFranchise'] == True) & (df['allStar'] == False)]
# nba_teams_df = df[['id', 'code', 'name']]
#########################################################

In [78]:
# display all nba teams with code
display(df)

,id,name,nickname,code,city,logo,allStar,nbaFranchise,leagues
0,1,Atlanta Hawks,Hawks,ATL,Atlanta,https://upload.wikimedia.org/wikipedia/fr/e/ee...,False,True,"{'standard': {'conference': 'East', 'division'..."
1,2,Boston Celtics,Celtics,BOS,Boston,https://upload.wikimedia.org/wikipedia/fr/thum...,False,True,"{'standard': {'conference': 'East', 'division'..."
3,4,Brooklyn Nets,Nets,BKN,Brooklyn,https://upload.wikimedia.org/wikipedia/commons...,False,True,"{'standard': {'conference': 'East', 'division'..."
4,5,Charlotte Hornets,Hornets,CHA,Charlotte,https://upload.wikimedia.org/wikipedia/fr/thum...,False,True,"{'standard': {'conference': 'East', 'division'..."
5,6,Chicago Bulls,Bulls,CHI,Chicago,https://upload.wikimedia.org/wikipedia/fr/thum...,False,True,"{'standard': {'conference': 'East', 'division'..."
6,7,Cleveland Cavaliers,Cavaliers,CLE,Cleveland,https://upload.wikimedia.org/wikipedia/fr/thum...,False,True,"{'standard': {'conference': 'East', 'division'..."
7,8,Dallas Mavericks,Mavericks,DAL,Dallas,https://upload.wikimedia.org/wikipedia/fr/thum...,False,True,"{'standard': {'conference': 'West', 'division'..."
8,9,Denver Nuggets,Nuggets,DEN,Denver,https://upload.wikimedia.org/wikipedia/fr/thum...,False,True,"{'standard': {'conference': 'West', 'division'..."
9,10,Detroit Pistons,Pistons,DET,Detroit,https://upload.wikimedia.org/wikipedia/commons...,False,True,"{'standard': {'conference': 'East', 'division'..."
10,11,Golden State Warriors,Warriors,GSW,Golden State,https://upload.wikimedia.org/wikipedia/fr/thum...,False,True,"{'standard': {'conference': 'West', 'division'..."


In [41]:
#########################################################
# games by game ids #####################################
def get_games_by_game_ids(season, team):
    url = "https://api-nba-v1.p.rapidapi.com/games"
    querystring = {"season":season,"team":team}

    response = requests.get(url, headers=headers, params=querystring)
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
    df = df.loc[(df["status.long"] == "Finished")]
    df = df.sort_values(by=["date.start"])
    
    # adding win col to df
    df['win'] = ''

    df.loc[
        ((df["scores.home.points"] > df["scores.visitors.points"]) & 
        (int(team) == df["teams.home.id"])) |
        ((df["scores.home.points"] < df["scores.visitors.points"]) & 
        (int(team) == df["teams.visitors.id"])),
        'win'
    ] = 1
    
    df.loc[
        ((df["scores.home.points"] < df["scores.visitors.points"]) & 
        (int(team) == df["teams.home.id"])) |
        ((df["scores.home.points"] > df["scores.visitors.points"]) & 
        (int(team) == df["teams.visitors.id"])),
        'win'
    ] = 0
    
    # adding home col to df
    df['home'] = ''
    
    df.loc[(int(team) == df["teams.home.id"]), 'home'] = 1
        
    df.loc[(int(team) == df["teams.visitors.id"]), 'home'] = 0    
    
    return drop_cols(df, cols_to_drop_for_game_stats)
#########################################################

In [25]:
#########################################################
# drop all cols from a df ##############################
def drop_cols(df, cols):
    for col in cols:
        df = df.drop(col, axis=1)
    return df
#########################################################

In [43]:
#########################################################
# get top n player stats by game by #####################
def get_top_players_per_game_df(n, team, game_id):    
    url = "https://api-nba-v1.p.rapidapi.com/players/statistics"

    querystring = {"game": game_id }
    
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )

    df_opponent = df.loc[df['team.id'] != team]
    df = df.loc[df['team.id'] == team]
    
    df_opponent = drop_cols(df_opponent, cols_to_drop_for_player_stats)
    df = drop_cols(df, cols_to_drop_for_player_stats)
    
    df_opponent = zero_non_numeric_values(df_opponent)
    df = zero_non_numeric_values(df)
    
    df_opponent = df_opponent.add_prefix("opponent.")

    return df_opponent.nlargest(n, "opponent.plusMinus"), df.nlargest(n, "plusMinus")
#########################################################

In [44]:
#########################################################
# flatten data frames ###################################
def flatten_df(df):
    # Flatten the DataFrame
    flattened_data = {}
    for col in df.columns:
        for row in range(df.shape[0]):
            new_col_name = f"{col}{row}"
            flattened_data[new_col_name] = df[col].iloc[row]

    # Convert to DataFrame
    return pd.DataFrame([flattened_data])
#########################################################

In [45]:
#########################################################
# per game, get top 5 players by plusMinus metric #######
def get_top_five_players_per_game(team_id, game_ids): 
    top_5_players_on_team_per_game = {}
    for game_id in game_ids:
        # transform player_stats_df
        opponent_top_five_player_stats, friendly_top_five_player_stats = get_top_players_per_game_df(5, team_id, game_id)
        opponent_top_five_player_stats = flatten_df(opponent_top_five_player_stats)
        friendly_top_five_player_stats = flatten_df(friendly_top_five_player_stats)
        top_five_players_stats = pd.concat(
            [opponent_top_five_player_stats, friendly_top_five_player_stats], 
            axis=1
        )
        ###########################
        top_5_players_on_team_per_game[game_id] = top_five_players_stats          
    return top_5_players_on_team_per_game
#########################################################

In [46]:
#########################################################
# get win prc, and last 10 win prc ######################
def get_win_prc(game_df):
    total_win_prc = []
    last_ten_win_prc = []
    last_ten_games_win_loss = []
    total_games_played = []
    last_ten_win_count = 0
    total_win_count = 0
    total_game_count = 0
    
    for index, row in game_df.iterrows():
        total_game_count += 1
        last_ten_games_win_loss.append(row['win'])
        
        if row['win']:
            total_win_count += 1
            last_ten_win_count += 1
            
        total_win_prc.append(total_win_count/total_game_count)
               
        if total_game_count >= 10:
            if last_ten_games_win_loss[0]:
                last_ten_win_count -= 1
            last_ten_games_win_loss.pop(0)
            last_ten_win_prc.append(last_ten_win_count/10)
        else:
            last_ten_win_prc.append(last_ten_win_count/total_game_count)
        
        total_games_played.append(total_game_count)            
    return last_ten_win_prc, total_win_prc, total_games_played
#########################################################

In [47]:
#########################################################
# combine player stats and games features ###############
import numpy as np
def combine_player_stats_and_games_data(games_df, player_stats_per_game):
    feature_map = []
    feature_map_cols_header = []
    for index, row in games_df.iterrows():
        game_id = row["id"]
        game_df = pd.DataFrame(row).transpose()
        players_stats_df = player_stats_per_game.get(game_id)
        if len(feature_map_cols_header) < 1:
            feature_map_cols_header = list(game_df) + list(players_stats_df)

        game_data = np.array(game_df.iloc[0])
        player_stats_data = np.array(players_stats_df.iloc[0])
        feature_map_data_row = np.concatenate((game_data, player_stats_data))
        feature_map.append(feature_map_data_row)
    return pd.DataFrame(feature_map, columns=feature_map_cols_header)
#########################################################

In [48]:
#########################################################
def get_feature_map_and_y(season, team_id):
    games_df = get_games_by_game_ids(season, team_id) 
    
    players_stats_per_games = get_top_five_players_per_game(
        team_id, 
        games_df["id"].array
    )
    
    last_ten_win_prc, total_win_prc, total_games_played = get_win_prc(games_df)
    games_df = games_df.assign(last_ten_w_prc=last_ten_win_prc)
    games_df = games_df.assign(w_prc=total_win_prc)
    games_df = games_df.assign(games_played=total_games_played)
    
    df = combine_player_stats_and_games_data(games_df, players_stats_per_games)
    df = zero_non_numeric_values(df)
    x = drop_cols(df, ["win", "id"])
    y = df["win"]
    return x, y 
#########################################################

In [49]:
def get_rf_regressor_and_classifier_model(x, y):
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
    # Regressor #############################################
    rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)

    # Train the model on the training set
    rf_regressor.fit(x_train, y_train)

    # Make predictions
    rf_regressor_predictions = rf_regressor.predict(x_test)
    
    # Evaluate the model
    rf_regressor_mse = mean_squared_error(y_test, rf_regressor_predictions)
    print(f'FR Regressor Mean Squared Error: {rf_regressor_mse}')
    #########################################################

    # Classifier ############################################
    # training random forest classifier 
    rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

    # Train the model on the training set
    rf_classifier.fit(x_train, y_train)
    
    # Make predictions on the test set
    rf_classifier_predictions = rf_classifier.predict(x_test)

    # Evaluate the model
    rf_classifier_accuracy = accuracy_score(y_test, rf_classifier_predictions)
    print(f'Accuracy: {rf_classifier_accuracy:.2f}')
    #########################################################
    
    return rf_regressor, rf_classifier


In [50]:
#########################################################
def zero_non_numeric_values(df):
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    return df
#########################################################

In [51]:
#########################################################
def get_data_and_rf_models(season, team_id): 
    x, y = get_feature_map_and_y(season, team_id)
    regressor, classifier = get_rf_regressor_and_classifier_model(x, y)
    return x, y, regressor, classifier
#########################################################

In [30]:
#########################################################
def get_player_season_stats_avgs(season, player_id):
    url = "https://api-nba-v1.p.rapidapi.com/players/statistics"
    querystring = {"id":player_id,"season":season}
    
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
    
    df = drop_cols(df, cols_to_drop_for_player_stats)
    df = zero_non_numeric_values(df) 
    
    return df.mean()
#########################################################
#########################################################
def get_players_season_stats_avgs(season, player_ids):
    player_stats_avgs = []
    for i in player_ids:
        player_stats_avgs.append(get_player_season_stats_avgs(season, i))
    
    df = pd.DataFrame(player_stats_avgs)
    return df.sort_values(by=["plusMinus"], ascending=False)
#########################################################


In [76]:
#########################################################
def get_player_lineups_for_tn():
    url = 'https://www.rotowire.com/basketball/nba-lineups.php'
    response = requests.get(url, verify=False)

    players_by_team = {}
    button_divs = soup.find_all("button", class_="see-court-on-off")

    for div in button_divs:
        nickname = div["data-nickname"]
        players_by_team.update({ nickname: [] })
        player_ids = div["data-lineup"].split(",")[:5]
        for player_id in player_ids:
            player_divs = soup.find_all("li", class_="lineup__player is-pct-play-100")
            for player_div in player_divs:
                a_tags = player_div.find_all('a', href=lambda href: href and player_id in href)
                for a_tag in a_tags:
                    players_by_team[nickname].append(a_tag["title"])
            
    return players_by_team
#########################################################

In [ ]:
#########################################################
def get_players_by_team_season(team_id, season):
    querystring = {"team":team_id,"season":season}
    return df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
#########################################################

In [ ]:
#########################################################
def get_team_latest_stats(team_id, season):
    url = "https://api-nba-v1.p.rapidapi.com/teams/statistics"
    querystring = {"id": team_id,"season": season}
    return df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
#########################################################

In [61]:
#########################################################
def get_prediction_feature_map(
    training_data_tail,
    season,
    team_nickname,
    o_team_nickname,
    home,
    players_name_map
):   
    url = "https://api-nba-v1.p.rapidapi.com/players"
    
    # getting player attributes
    df = get_players_by_and_team_season(team_id, season)
    
    o_df = get_players_by_and_team_season(o_team_id, season)

    concat_names = df['firstname'] + " " + df['lastname']
    o_concat_names = o_df['firstname'] + " " + o_df['lastname']
    
    name_list = players_name_map.get(team_nickname)
    o_name_list = players_name_map.get(o_nickname)
    
    # Filter the DataFrame based on whether the concatenated names exist in name sets
    df_players = df[concat_names.isin(set(name_set))] 
    o_df_players = o_df[o_concat_names.isin(set(o_name_list))]
    
    # TODO get team ppg and o team ppg
    get_team_latest_stats()
    
    player_ids = np.array(df["id"])
    o_player_ids = np.array(o_df["id"])
    
    players_stats_avgs = get_players_season_stats_avgs(season, player_ids)
    o_players_stats_avgs = get_players_season_stats_avgs(season, o_player_ids)
    o_players_stats_avgs = o_players_stats_avgs.add_prefix("opponent.")

    o_players_stats_avgs = flatten_df(o_players_stats_avgs)
    players_stats_avgs = flatten_df(players_stats_avgs)

    players_stats = pd.concat(
        [o_players_stats_avgs, players_stats_avgs], 
        axis=1
    )
    
    games_col_headers = [
        "scores.visitors.points",
        "scores.home.points", 
        "home", 
        "last_ten_w_prc", 
        "w_prc", 
        "games_played"
    ]
    
    games_data = [
        visitors_avg_ppg, 
        home_avg_ppg, 
        home, 
        training_data_tail["last_ten_w_prc"].iloc[0], 
        training_data_tail["w_prc"].iloc[0],
        training_data_tail["games_played"].iloc[0] + 1
    ]
        
    feature_map_col_headers = games_col_headers + list(players_stats)
    feature_map_data_row = np.concatenate((games_data, players_stats.iloc[0]))
    return pd.DataFrame([feature_map_data_row], columns=feature_map_col_headers)    
#########################################################

In [54]:
# Get models for teams ##################################
print("Sixers")
sixers_x, sixers_y, sixers_regressor, sixers_classifier = get_data_and_rf_models("2023", 27)
print()

print("Knicks")
knicks_x, knicks_y, knicks_regressor, knicks_classifier = get_data_and_rf_models("2023", 24)
print()

print("Pacers")
pacers_x, pacers_y, pacers_regressor, pacers_classifier = get_data_and_rf_models("2023", 15)
print()

print("Thunder")
okc_x, okc_y, okc_regressor, okc_classifier = get_data_and_rf_models("2023", 25)
print()

print("Wiz")
wiz_x, wiz_y, wiz_regressor, wiz_classifier = get_data_and_rf_models("2023", 41)
print()

print("Grizzlies")
grizz_x, grizz_y, grizz_regressor, grizz_classifier = get_data_and_rf_models("2023", 19)
print()

print("Rockets")
rockets_x, rockets_y, rockets_regressor, rockets_classifier = get_data_and_rf_models("2023", 14)
print()

print("Spurs")
spurs_x, spurs_y, spurs_regressor, spurs_classifier = get_data_and_rf_models("2023", 31)
print()

print("Celtics")
bos_x, bos_y, bos_regressor, bos_classifier = get_data_and_rf_models("2023", 2)
print()

print("Jazz")
jazz_x, jazz_y, jazz_regressor, jazz_classifier = get_data_and_rf_models("2023", 40)
print()

print("Wolves")
wolves_x, wolves_y, wolves_regressor, wolves_classifier = get_data_and_rf_models("2023", 22)
print()

print("Clippers")
lac_x, lac_y, lac_regressor, lac_classifier = get_data_and_rf_models("2023", 16)
print()

print("Bucks")
bucks_x, bucks_y, bucks_regressor, bucks_classifier = get_data_and_rf_models("2023", 21)
print()

print("Kings")
kings_x, kings_y, kings_regressor, kings_classifier = get_data_and_rf_models("2023", 30)
print()

# print("Raptors")
# rapt_x, rapt_y, rapt_regressor, rapt_classifier = get_data_and_rf_models("2023", 38)
# print()

# print("Pistons")
# pistons_x, pistons_y, pistons_regressor, pistons_classifier = get_data_and_rf_models("2023", 10)
# print()

# print("Nets")
# nets_x, nets_y, nets_regressor, nets_classifier = get_data_and_rf_models("2023", 4)
# print()

# print("Magic")
# magic_x, magic_y, magic_regressor, magic_classifier = get_data_and_rf_models("2023", 26)
# print()

# print("Nuggets")
# nugs_x, nugs_y, nugs_regressor, nugs_classifier = get_data_and_rf_models("2023", 9)
# print()

# print("Heat")
# heat_x, heat_y, heat_regressor, heat_classifier = get_data_and_rf_models("2023", 20)
# print()

# print("Bulls")
# bulls_x, bulls_y, bulls_regressor, bulls_classifier = get_data_and_rf_models("2023", 6)
# print()

# print("Hornets")
# hornets_x, hornets_y, hornets_regressor, hornets_classifier = get_data_and_models("2023", 5)
# print()

# print("Cavs")
# cavs_x, cavs_y, cavs_regressor, cavs_classifier = get_data_and_rf_models("2023", 7)
# print()

# print("Pelicans")
# pels_x, pels_y, pels_regressor, pels_classifier = get_data_and_rf_models("2023", 23)
# print()

# print("Warriors")
# war_x, war_y, war_regressor, war_classifier = get_data_and_rf_models("2023", 11)
# print()

# print("Suns")
# suns_x, sun_y, suns_regressor, suns_classifier = get_data_and_rf_models("2023", 28)
# print()

# print("Mavericks")
# mavs_x, mavs_y, mavs_regressor, mavs_classifier = get_data_and_rf_models("2023", 8)
# print()

# print("Lakers")
# lal_x, lal_y, lal_regressor, lal_classifier = get_data_and_rf_models("2023", 17)
# print()

# print("Kings")
# lal_x, lal_y, lal_regressor, lal_classifier = get_data_and_rf_models("2023", 30)
# print()

# print("Hawks")
# atl_x, atl_y, atl_regressor, atl_classifier = get_data_and_rf_models("2023", 1)
# print()

# print("Blazers")
# blazers_x, blazers_y, blazers_regressor, blazers_classifier = get_data_and_rf_models("2023", 29)
# print()
#########################################################

Sixers
FR Regressor Mean Squared Error: 0.029571428571428564
Accuracy: 0.93

Knicks
FR Regressor Mean Squared Error: 0.05872142857142857
Accuracy: 1.00

Pacers
FR Regressor Mean Squared Error: 0.08690714285714285
Accuracy: 1.00

Thunder
FR Regressor Mean Squared Error: 0.09195
Accuracy: 0.93

Wiz
FR Regressor Mean Squared Error: 0.014685714285714283
Accuracy: 1.00

Grizzlies
FR Regressor Mean Squared Error: 0.12665714285714288
Accuracy: 0.93

Rockets
FR Regressor Mean Squared Error: 0.05182142857142859
Accuracy: 0.86

Spurs
FR Regressor Mean Squared Error: 0.13545
Accuracy: 0.93

Celtics
FR Regressor Mean Squared Error: 0.10752857142857142
Accuracy: 0.93

Jazz
FR Regressor Mean Squared Error: 0.040557142857142854
Accuracy: 0.93

Wolves
FR Regressor Mean Squared Error: 0.041
Accuracy: 0.93

Clippers
FR Regressor Mean Squared Error: 0.18202857142857143
Accuracy: 0.93

Bucks
FR Regressor Mean Squared Error: 0.038528571428571426
Accuracy: 1.00

Kings
FR Regressor Mean Squared Error: 0.0418

In [62]:
#########################################################
# View Predictions ######################################

In [64]:
sixers_pred = get_prediction_feature_map(
    sixers_x.tail(1),
    "2023",
    27,
    24,
    112.4,
    115.9,
    0,
    "Kyle Lowry",
    "Tyrese Maxey",
    "Kelly Oubre Jr.",
    "Tobias Harris",
    "Mo Bamba",
    "Jalen Brunson",
    "Donte DiVincenzo",
    "Josh Hart",
    "Precious Achiuwa",
    "Isaiah Hartenstein",
)

knicks_pred = get_prediction_feature_map(
    knicks_x.tail(1),
    "2023",
    24,
    27,
    112.4,
    115.9,
    1,
    "Jalen Brunson",
    "Donte DiVincenzo",
    "Josh Hart",
    "Precious Achiuwa",
    "Isaiah Hartenstein",
    "Kyle Lowry",
    "Tyrese Maxey",
    "Kelly Oubre Jr.",
    "Tobias Harris",
    "Mo Bamba",
)

In [65]:
print("sixers regressor : ", sixers_regressor.predict(sixers_pred))
print("knicks regressor : ", knicks_regressor.predict(knicks_pred))
print("sixers classifier : ", sixers_classifier.predict(sixers_pred))
print("knicks classifier : ", knicks_classifier.predict(knicks_pred))

sixers regressor :  [0.59]
knicks regressor :  [0.82]
sixers classifier :  [0]
knicks classifier :  [0]


In [66]:
pacers_pred = get_prediction_feature_map(
    pacers_x.tail(1),
    "2023",
    15,
    25,
    120.8,
    123.1,
    0,
    "Tyrese Haliburton",
    "Andrew Nembhard",
    "Aaron Nesmith",
    "Pascal Siakam",
    "Myles Turner",
    "Shai Gilgeous-Alexander",
    "Josh Giddey",
    "Luguentz Dort",
    "Gordon Hayward",
    "Chet Holmgren",
)

okc_pred = get_prediction_feature_map(
    okc_x.tail(1),
    "2023",
    25,
    15,
    120.8,
    123.1,
    1,
    "Shai Gilgeous-Alexander",
    "Josh Giddey",
    "Luguentz Dort",
    "Gordon Hayward",
    "Chet Holmgren",
    "Tyrese Haliburton",
    "Andrew Nembhard",
    "Aaron Nesmith",
    "Pascal Siakam",
    "Myles Turner",
)

In [67]:
print("pacers regressor : ", pacers_regressor.predict(pacers_pred))
print("okc regressor : ", okc_regressor.predict(okc_pred))
print("pacers classifier : ", pacers_classifier.predict(pacers_pred))
print("okc classifier : ", okc_classifier.predict(okc_pred))

pacers regressor :  [0.73]
okc regressor :  [0.79]
pacers classifier :  [1]
okc classifier :  [1]


In [68]:
wiz_pred = get_prediction_feature_map(
    wiz_x.tail(1),
    "2023",
    41,
    19,
    105.8,
    114.4,
    0,
    "Tyus Jones",
    "Bilal Coulibaly",
    "Deni Avdija",
    "Kyle Kuzma",
    "Eugene Omoruyi",
    "Luke Kennard",
    "John Konchar",
    "Jake LaRavia",
    "Santi Aldama",
    "Trey Jemison"
)

grizz_pred = get_prediction_feature_map(
    grizz_x.tail(1),
    "2023",
    19,
    41,
    105.8,
    114.4,
    1,
    "Luke Kennard",
    "John Konchar",
    "Jake LaRavia",
    "Santi Aldama",
    "Trey Jemison",
    "Tyus Jones",
    "Bilal Coulibaly",
    "Deni Avdija",
    "Kyle Kuzma",
    "Eugene Omoruyi",
)

In [70]:
print("wiz regressor : ", wiz_regressor.predict(wiz_pred))
print("grizz regressor : ", grizz_regressor.predict(grizz_pred))
print("wiz classifier : ", wiz_classifier.predict(wiz_pred))
print("grizz classifier : ", grizz_classifier.predict(grizz_pred))

wiz regressor :  [0.42]
grizz regressor :  [0.22]
wiz classifier :  [0]
grizz classifier :  [0]


In [72]:
rockets_pred = get_prediction_feature_map(
    rockets_x.tail(1),
    "2023",
    14,
    31,
    112.6,
    113.0,
    0,
    "Fred VanVleet",
    "Jalen Green",
    "Dillon Brooks",
    "Jabari Smith Jr.",
    "Jock Landale",
    "Tre Jones",
    "Devin Vassell",
    "Julian Champagnie",
    "Jeremy Sochan",
    "Victor Wembanyama"
)

spurs_pred = get_prediction_feature_map(
    spurs_x.tail(1),
    "2023",
    31,
    14,
    112.6,
    113.0,
    1,
    "Tre Jones",
    "Devin Vassell",
    "Julian Champagnie",
    "Jeremy Sochan",
    "Victor Wembanyama",
    "Fred VanVleet",
    "Jalen Green",
    "Dillon Brooks",
    "Jabari Smith Jr.",
    "Jock Landale",
)

In [74]:
print("rockets regressor : ", rockets_regressor.predict(rockets_pred))
print("spurs regressor : ", spurs_regressor.predict(spurs_pred))
print("rockets classifier : ", rockets_classifier.predict(rockets_pred))
print("spurs classifier : ", spurs_classifier.predict(spurs_pred))

rockets regressor :  [0.43]
spurs regressor :  [0.95]
rockets classifier :  [0]
spurs classifier :  [0]


In [75]:
bos_pred = get_prediction_feature_map(
    bos_x.tail(1),
    "2023",
    2,
    40,
    117.7,
    120.7,
    0,
    "Payton Pritchard",
    "Derrick White",
    "Jaylen Brown",
    "Jayson Tatum",
    "Al Horford",
    "Keyonte George",
    "Collin Sexton",
    "Brice Sensabaugh",
    "Luka Samanic",
    "John Collins",
)

jazz_pred = get_prediction_feature_map(
    jazz_x.tail(1),
    "2023",
    40,
    2,
    117.7,
    120.7,
    1,
    "Keyonte George",
    "Collin Sexton",
    "Brice Sensabaugh",
    "Luka Samanic",
    "John Collins",
    "Payton Pritchard",
    "Derrick White",
    "Jaylen Brown",
    "Jayson Tatum",
    "Al Horford"
)

In [76]:
print("bos regressor : ", bos_regressor.predict(bos_pred))
print("jazz regressor : ", jazz_regressor.predict(jazz_pred))
print("bos classifier : ", bos_classifier.predict(bos_pred))
print("jazz classifier : ", jazz_classifier.predict(jazz_pred))

bos regressor :  [0.95]
jazz regressor :  [0.]
bos classifier :  [1]
jazz classifier :  [0]


In [77]:
wolves_pred = get_prediction_feature_map(
    wolves_x.tail(1),
    "2023",
    22,
    16,
    117.2,
    113.1,
    0,
    "Mike Conley",
    "Anthony Edwards",
    "Jaden McDaniels",
    "Kyle Anderson",
    "Rudy Gobert",
    "James Harden",
    "Terance Mann",
    "Paul George",
    "Kawhi Leonard",
    "Ivica Zubac"
)

lac_pred = get_prediction_feature_map(
    lac_x.tail(1),
    "2023",
    16,
    22,
    117.2,
    113.1,
    1,
    "James Harden",
    "Terance Mann",
    "Paul George",
    "Kawhi Leonard",
    "Ivica Zubac",
    "Mike Conley",
    "Anthony Edwards",
    "Jaden McDaniels",
    "Kyle Anderson",
    "Rudy Gobert",
)

In [78]:
print("wolves regressor : ", wolves_regressor.predict(wolves_pred))
print("lac regressor : ", lac_regressor.predict(lac_pred))
print("wolves classifier : ", wolves_classifier.predict(wolves_pred))
print("lac classifier : ", lac_classifier.predict(lac_pred))

wolves regressor :  [0.64]
lac regressor :  [0.68]
wolves classifier :  [1]
lac classifier :  [1]


In [79]:
bucks_pred = get_prediction_feature_map(
    bucks_x.tail(1),
    "2023",
    21,
    30,
    118.2,
    120.9,
    0,
    "Damian Lillard",
    "Malik Beasley",
    "Jae Crowder",
    "Giannis Antetokounmpo",
    "Brook Lopez",
    "De'Aaron Fox",
    "Kevin Huerter",
    "Harrison Barnes",
    "Keegan Murray",
    "Domantas Sabonis"
)

kings_pred = get_prediction_feature_map(
    kings_x.tail(1),
    "2023",
    30,
    21,
    118.2,
    120.9,
    1,
    "De'Aaron Fox",
    "Kevin Huerter",
    "Harrison Barnes",
    "Keegan Murray",
    "Domantas Sabonis",
    "Damian Lillard",
    "Malik Beasley",
    "Jae Crowder",
    "Giannis Antetokounmpo",
    "Brook Lopez"
)

In [80]:
print("bucks regressor : ", bucks_regressor.predict(bucks_pred))
print("kings regressor : ", kings_regressor.predict(kings_pred))
print("bucks classifier : ", bucks_classifier.predict(bucks_pred))
print("kings classifier : ", kings_classifier.predict(kings_pred))

bucks regressor :  [0.75]
kings regressor :  [0.33]
bucks classifier :  [1]
kings classifier :  [0]


In [ ]:
# rapt_pred = get_prediction_feature_map(
#     rapt_x.tail(1),
#     "2023",
#     27,
#     10,
#     112.3,
#     114.2,
#     0,
#     "Bruce Brown",
#     "RJ Barrett",
#     "Kelly Olynyk",
#     "Gradey Dick",
#     "Ochai Agbaji",
#     "Cade Cunningham",
#     "Jaden Ivey",
#     "Evan Fournier",
#     "Isaiah Stewart",
#     "Jalen Duren",
# )

# pistons_pred = get_prediction_feature_map(
#     pistons_x.tail(1),
#     "2023",
#     10,
#     38,
#     112.3,
#     114.2,
#     1,
#     "Cade Cunningham",
#     "Jaden Ivey",
#     "Evan Fournier",
#     "Isaiah Stewart",
#     "Jalen Duren",
#     "Bruce Brown",
#     "RJ Barrett",
#     "Kelly Olynyk",
#     "Gradey Dick",
#     "Ochai Agbaji",
# )

In [ ]:
# print("raptors regressor : ", rapt_regressor.predict(rapt_pred))
# print("pistons regressor : ", pistons_regressor.predict(pistons_pred))
# print("raptors classifier : ", rapt_classifier.predict(rapt_pred))
# print("pistons classifier : ", pistons_classifier.predict(pistons_pred))

In [ ]:
# nets_pred = get_prediction_feature_map(
#     nets_x.tail(1),
#     "2023",
#     4,
#     26,
#     110.6,
#     111.9,
#     0,
#     "Dennis Smith Jr.",
#     "Cam Thomas",
#     "Mikal Bridges",
#     "Dorian Finney-Smith",
#     "Nic Claxton",
#     "Cole Anthony",
#     "Gary Harris",
#     "Franz Wagner",
#     "Paolo Banchero",
#     "Wendell Carter Jr."
# )

# magic_pred = get_prediction_feature_map(
#     magic_x.tail(1),
#     "2023",
#     26,
#     4,
#      110.6,
#     111.9,
#     1,
#     "Cole Anthony",
#     "Gary Harris",
#     "Franz Wagner",
#     "Paolo Banchero",
#     "Wendell Carter Jr.",
#     "Dennis Smith Jr.",
#     "Cam Thomas",
#     "Mikal Bridges",
#     "Dorian Finney-Smith",
#     "Nic Claxton"
# )

In [ ]:
# print("nets regressor : ", nets_regressor.predict(nets_pred))
# print("magic regressor : ", magic_regressor.predict(magic_pred))
# print("nets classifier : ", nets_classifier.predict(nets_pred))
# print("magic classifier : ", magic_classifier.predict(magic_pred))

In [ ]:
# nugs_pred = get_prediction_feature_map(
#     nugs_x.tail(1),
#     "2023",
#     9,
#     20,
#     110.5,
#     114.9,
#     0,
#     "Jamal Murray",
#     "Kentavious Caldwell-Pope",
#     "Michael Porter Jr.",
#     "Aaron Gordon",
#     "Nikola Jokic",
#     "Terry Rozier",
#     "Duncan Robinson",
#     "Jimmy Butler",
#     "Nikola Jovic",
#     "Bam Adebayo"
# )

# heat_pred = get_prediction_feature_map(
#     heat_x.tail(1),
#     "2023",
#     20,
#     9,
#     110.5,
#     114.9,
#     1,
#     "Terry Rozier",
#     "Duncan Robinson",
#     "Jimmy Butler",
#     "Nikola Jovic",
#     "Bam Adebayo",
#     "Jamal Murray",
#     "Kentavious Caldwell-Pope",
#     "Michael Porter Jr.",
#     "Aaron Gordon",
#     "Nikola Jokic",
# )

In [ ]:
# print("nugs classifier : ", nugs_classifier.predict(nugs_pred))
# print("heat classifier : ", heat_classifier.predict(heat_pred))
# print("nugs regressor : ", nugs_regressor.predict(nugs_pred))
# print("heat regressor : ", heat_regressor.predict(heat_pred))

In [ ]:
# print("7")
# mavs_pred = get_prediction_feature_map(
#     mavs_x.tail(1),
#     "2023",
#     8,
#     6,
#     111.8,
#     119,
#     0,
#     "Luka Doncic",
#     "Kyrie Irving",
#     "Derrick Jones Jr.",
#     "P.J. Washington",
#     "Daniel Gafford",
#     "Coby White",
#     "Ayo Dosunmu",
#     "Alex Caruso",
#     "DeMar DeRozan",
#     "Nikola Vucevic"
# )

# print("8")
# bulls_pred = get_prediction_feature_map(
#     bulls_x.tail(1),
#     "2023",
#     6,
#     8,
#     111.8,
#     119,
#     1,
#     "Coby White",
#     "Ayo Dosunmu",
#     "Alex Caruso",
#     "DeMar DeRozan",
#     "Nikola Vucevic",
#     "Luka Doncic",
#     "Kyrie Irving",
#     "Derrick Jones Jr.",
#     "P.J. Washington",
#     "Daniel Gafford"
# )

In [ ]:
# print("mavs classifier : ", mavs_classifier.predict(mavs_pred))
# print("bulls classifier : ", bulls_classifier.predict(bulls_pred))
# print("mavs regressor : ", mavs_regressor.predict(mavs_pred))
# print("bulls regressor : ", bulls_regressor.predict(bulls_pred))

In [ ]:
# rapt_pred = get_prediction_feature_map(
#     rapt_x.tail(1),
#     "2023",
#     38,
#     9,
#     114.9,
#     114.2,
#     0,
#     "Bruce Brown",
#     "RJ Barrett",
#     "Kelly Olynyk",
#     "Gradey Dick",
#     "Ochai Agbaji",
#     "Jamal Murray",
#     "Kentavious Caldwell-Pope",
#     "Michael Porter Jr.",
#     "Aaron Gordon",
#     "Nikola Jokic"
# )

# nugs_pred = get_prediction_feature_map(
#     nugs_x.tail(1),
#     "2023",
#     9,
#     38,
#     114.9,
#     114.2,
#     1,
#     "Jamal Murray",
#     "Kentavious Caldwell-Pope",
#     "Michael Porter Jr.",
#     "Aaron Gordon",
#     "Nikola Jokic",
#     "Bruce Brown",
#     "RJ Barrett",
#     "Kelly Olynyk",
#     "Gradey Dick",
#     "Ochai Agbaji"
# )

In [ ]:
# print("rapt classifier: ", rapt_classifier.predict(rapt_pred))
# print("nugs classifier: ", nugs_classifier.predict(nugs_pred))
# print("rapt regressor: ", rapt_regressor.predict(rapt_pred))
# print("nugs regressor: ", nugs_regressor.predict(nugs_pred))

In [ ]:
# blazers_pred = get_prediction_feature_map(
#     blazers_x.tail(1),
#     "2023",
#     29,
#     2,
#     108.1,
#     120.7,
#     1,
#     "Scoot Henderson",
#     "Anfernee Simons",
#     "Jerami Grant",
#     "Toumani Camara",
#     "Deandre Ayton",
#     "Payton Pritchard",
#     "Derrick White",
#     "Jaylen Brown",
#     "Jayson Tatum",
#     "Kristaps Porzingis"
# )

# bos_pred = get_prediction_feature_map(
#     bos_x.tail(1),
#     "2023",
#     2,
#     29,
#     108.1,
#     120.7,
#     0,
#     "Payton Pritchard",
#     "Derrick White",
#     "Jaylen Brown",
#     "Jayson Tatum",
#     "Kristaps Porzingis",
#     "Scoot Henderson",
#     "Anfernee Simons",
#     "Jerami Grant",
#     "Toumani Camara",
#     "Deandre Ayton"
# )

In [ ]:
# print("blazers classifier: ", blazers_classifier.predict(blazers_pred))
# print("bos classifier: ", bos_classifier.predict(bos_pred))
# print("blazers regressor: ", blazers_regressor.predict(blazers_pred))
# print("bos regressor: ", bos_regressor.predict(bos_pred))

In [ ]:
# hornets_pred = get_prediction_feature_map(
#     hornets_x.tail(1),
#     "2023",
#     5,
#     10,
#     112.3,
#     107.2,
#     0,
#     "Vasilije Micic",
#     "Grant Williams",
#     "Brandon Miller",
#     "Miles Bridges",
#     "Nick Richards",
#     "Cade Cunningham",
#     "Jaden Ivey",
#     "Evan Fournier",
#     "Isaiah Stewart",
#     "Jalen Duren"
# )

# suns_pred = get_prediction_feature_map(
#     suns_x.tail(1),
#     "2023",
#     28,
#     7,
#     113.8,
#     117,
#     0,
#     "Devin Booker",
#     "Bradley Beal",
#     "Grayson Allen",
#     "Kevin Durant",
#     "Jusuf Nurkic",
#     "Darius Garland",
#     "Caris LeVert",
#     "Isaac Okoro",
#     "George Niang",
#     "Jarrett Allen"
# )

# cavs_pred = get_prediction_feature_map(
#     cavs_x.tail(1),
#     "2023",
#     7,
#     28,
#     113.8,
#     117,
#     1,
#     "Darius Garland",
#     "Caris LeVert",
#     "Isaac Okoro",
#     "George Niang",
#     "Jarrett Allen",
#     "Devin Booker",
#     "Bradley Beal",
#     "Grayson Allen",
#     "Kevin Durant",
#     "Jusuf Nurkic",
# )

# war_pred = get_prediction_feature_map(
#     war_x.tail(1),
#     "2023",
#     11,
#     31,
#     112.6,
#     118.7,
#     0,
#     "Chris Paul",
#     "Brandin Podziemski",
#     "Andrew Wiggins",
#     "Jonathan Kuminga",
#     "Draymond Green",
#     "Tre Jones",
#     "Devin Vassell",
#     "Julian Champagnie",
#     "Jeremy Sochan",
#     "Victor Wembanyama",  
# )